## Databricks / R Data Pipeline

This Databricks notebook provides an example of a solution to migrating a process that uses both SQL and R. This deomnstrates the 'Move all code into Databricks' solution detailed in the workbook. _Currently this only involves running a notebook, however it could also be incorporated into a job - maybe update to add before finalising_

Pipeline stages:
1. **Check data** - Look at data structure of Autumn 2025 pupils and schools data
2. **Data manipulation** - Select columns and parse metadata, join pupils to schools
3. **Save intermediate output** - Save intermediate SQL output before further data processing in R 
4. **Outputs** - Produce average pupil premium by Ofsted rating summary

In [0]:
-- Define catalog being used
USE CATALOG catalog_40_copper_analyst_training;

### Check data
Look at data structure.

In [0]:
-- Check structure of pupils_autumn_2024 table
SELECT * FROM catalog_40_copper_analyst_training.bronze.pupils_autumn_2024 LIMIT 1000

In [0]:
-- Check structure of schools_autumn_2024 table
SELECT * FROM catalog_40_copper_analyst_training.bronze.schools_autumn_2024 LIMIT 1000

In [0]:
-- Check structure of assessments table
SELECT * FROM catalog_40_copper_analyst_training.bronze.assessments LIMIT 1000;

### Data Manipulation
Select required columns and parse JSON metadata, join pupils to schools.

In [0]:
-- Pupils table: select columns and parse JSON metadata
CREATE OR REPLACE TEMPORARY VIEW pupils_aut AS
SELECT 
  pupil_id,
  gender,
  school_urn,
  metadata_json:address:postcode AS postcode,
  LOWER(metadata_json:sen_status) AS sen_status,
  metadata_json:fsm_eligible AS fsm_status
FROM catalog_40_copper_analyst_training.bronze.pupils_autumn_2024;

-- Schools table: select columns, parse JSON, reformat data in city column, deduplicate
CREATE OR REPLACE TEMP VIEW schools_aut AS -- We use TEMP here instead of TEMPORARY in the above example - these can be used interchangeably
SELECT DISTINCT
  school_urn,
  INITCAP(city) AS city,
  school_type,
  REPLACE(metadata_json:ofsted_rating, '"', '') AS ofsted_rating,
  CAST(metadata_json:pupil_premium_pct AS DOUBLE) AS pupil_premium_pct
FROM catalog_40_copper_analyst_training.bronze.schools_autumn_2024;

-- Assessments table: select columns, standardise subject, deduplicate
CREATE OR REPLACE TEMP VIEW assessments_aut AS -- need to create a temporary view here as tables are not permitted with dedicated compute?
SELECT DISTINCT
  assessment_id,
  pupil_id,
  school_urn,
  CASE 
    WHEN LOWER(subject) LIKE '%e' THEN 'English'
    WHEN LOWER(subject) LIKE '%m' THEN 'Maths'
    WHEN LOWER(subject) LIKE '%s' THEN 'Science' 
  ELSE 'Unknown' END AS subject,
  score,
  grade
FROM catalog_40_copper_analyst_training.bronze.assessments
WHERE term = 'Autumn 2024';

-- Join pupils, assessments, and schools tables ahead of performing calculations
CREATE OR REPLACE TEMP VIEW assessments_schools_aut AS -- need to create a temporary view here as tables are not permitted with dedicated compute?
SELECT 
  p.*,
  s.city,
  s.school_type,
  s.ofsted_rating,
  s.pupil_premium_pct,
  a.assessment_id,
  a.subject,
  a.score,
  a.grade
FROM pupils_aut p
LEFT JOIN assessments_aut a
  ON p.pupil_id = a.pupil_id
LEFT JOIN schools_aut s
  ON p.school_urn = s.school_urn;

-- Save intermediate output to schema with write permissions
CREATE OR REPLACE TABLE catalog_40_copper_statistics_services.rtarrant.assessments_schools_aut
AS
SELECT *
FROM assessments_schools_aut

### Save intermediate
In order to use the intermediate table we've created, we need to save it as a CSV, before reading this CSV in R in our second notebook. Since we can't write to the analyst traininng catalog, you will need to save this to your team's catalog, under a schema called 'your_name'. We will use pyspark to create the schema if it doesn't exist, and then save the table.

In [0]:
%py
# Create volume if it doesn't exist
# Make sure to change this to your team's catalog, and your name!
catalog = "catalog_40_copper_statistics_services"
schema = "rtarrant"
volume_name = "pp_analysis"

spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume_name}")

# convert table to spark table
df = spark.table("assessments_schools_aut")

# write CSV (note - I think this only works for small csvs?)
df.toPandas().to_csv("/Volumes/catalog_40_copper_statistics_services/rtarrant/pp_analysis/assessments_schools_aut.csv", index=False)